In [2]:
import os
import io
import json
import time
import argparse
import sqlite3
import logging
import datetime
 
import requests
import pandas as pd
 
DB_PATH = "data/processed/east_coast_gas.db"
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
 
CITIES = {
    "Sydney":    {"lat": -33.87, "lon": 151.21},
    "Brisbane":  {"lat": -27.47, "lon": 153.03},
    "Adelaide":  {"lat": -34.93, "lon": 138.60},
    "Melbourne": {"lat": -37.81, "lon": 144.96},
}
HDD_BASE = 18.0
HEADERS = {"User-Agent": "Mozilla/5.0"}
 
# DWGM 历史参考价单文件：price_bod_gst_ex 列 = 6am/BOD 价
DWGM_INT041_URL = "https://nemweb.com.au/Reports/CURRENT/VicGas/int041_v4_market_and_reference_prices_1.csv"
 
 
def fetch(url, tries=3):
    for i in range(tries):
        try:
            r = requests.get(url, headers=HEADERS, timeout=30)
            r.raise_for_status()
            return r.content
        except requests.exceptions.RequestException as e:
            logging.warning(f"fetch attempt {i+1} failed: {url} :: {e}")
            if i < tries - 1:
                time.sleep(2 ** i)
    return None
 
 
# ============================================================
# 天气回填：Open-Meteo archive (观测 HDD)
# ============================================================
def backfill_weather(start_date, end_date=None):
    end_date = end_date or (datetime.date.today() - datetime.timedelta(days=1)).isoformat()
    ingested_at = datetime.datetime.now(datetime.timezone.utc).isoformat()
    records = []
 
    for city, geo in CITIES.items():
        url = (
            "https://archive-api.open-meteo.com/v1/archive"
            f"?latitude={geo['lat']}&longitude={geo['lon']}"
            f"&start_date={start_date}&end_date={end_date}"
            "&daily=temperature_2m_max,temperature_2m_min"
            "&timezone=Australia%2FSydney"
        )
        content = fetch(url)
        if not content:
            logging.error(f"weather backfill failed: {city}")
            continue
        try:
            data = json.loads(content)
            daily = data["daily"]
            for i, d in enumerate(daily["time"]):
                mx, mn = daily["temperature_2m_max"][i], daily["temperature_2m_min"][i]
                tmean = None if (mx is None or mn is None) else (mx + mn) / 2.0
                hdd = None if tmean is None else max(0.0, HDD_BASE - tmean)
                records.append((
                    d, city,
                    None if mx is None else float(mx),
                    None if mn is None else float(mn),
                    None if tmean is None else float(tmean),
                    None if hdd is None else float(hdd),
                    0,  # is_forecast=0：archive 观测真值
                    "open-meteo-archive", ingested_at,
                ))
            logging.info(f"weather {city}: {len(daily['time'])} days")
        except Exception as e:
            logging.exception(f"weather parse error {city}: {e}")
 
    if records:
        conn = sqlite3.connect(DB_PATH)
        conn.executemany(
            'INSERT OR REPLACE INTO weather '
            '(date, city, temp_max, temp_min, temp_mean, hdd, is_forecast, source, ingested_at) '
            'VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)', records)
        conn.commit(); conn.close()
        logging.info(f"Weather backfill saved: {len(records)} rows.")
    return len(records)
 
 
# ============================================================
# DWGM 回填：解析 int041 单文件，取 price_bod (6am/BOD) 价
# ============================================================
def backfill_dwgm():
    content = fetch(DWGM_INT041_URL)
    if not content:
        logging.error("DWGM int041 download failed.")
        return 0
 
    try:
        df = pd.read_csv(io.BytesIO(content))
    except Exception as e:
        logging.exception(f"int041 read failed: {e}")
        return 0
 
    needed = {'gas_date', 'price_bod_gst_ex', 'current_date'}
    missing = needed - set(df.columns)
    if missing:
        logging.error(f"int041 缺少必需列: {missing}。实际列: {list(df.columns)}")
        return 0
 
    ingested_at = datetime.datetime.now(datetime.timezone.utc).isoformat()
    source_file = "int041_v4_market_and_reference_prices_1.csv"
 
    df = df.copy()
    df['gas_date_dt'] = pd.to_datetime(df['gas_date'], dayfirst=True, errors='coerce')
    cd = pd.to_datetime(df['current_date'], dayfirst=True, errors='coerce').max()
    current_date = cd.normalize() if pd.notna(cd) else pd.Timestamp(datetime.date.today())
 
    n_bad = int(df['gas_date_dt'].isna().sum())
    if n_bad:
        logging.warning(f"int041: {n_bad} 行 gas_date 解析失败，已剔除。")
        df = df[df['gas_date_dt'].notna()]
 
    records = []
    for _, r in df.iterrows():
        gdate = r['gas_date_dt']
        price = r['price_bod_gst_ex']
        if pd.isna(price):
            continue
        is_forecast = 1 if gdate.normalize() > current_date else 0
        records.append((
            gdate.strftime('%Y-%m-%d'),
            float(price),
            "BOD (6am)",          # approval_datetime 位记口径标记，区别于 daily 真实时间戳
            is_forecast,
            source_file,
            ingested_at,
        ))
 
    if records:
        conn = sqlite3.connect(DB_PATH)
        conn.executemany(
            'INSERT OR REPLACE INTO dwgm_prices '
            '(gas_date, price_6am_schedule, approval_datetime, is_forecast, source_file, ingested_at) '
            'VALUES (?, ?, ?, ?, ?, ?)', records)
        conn.commit(); conn.close()
        logging.info(f"DWGM backfill saved: {len(records)} rows "
                     f"({records[0][0]} .. {records[-1][0]}).")
    return len(records)
 
 
# ============================================================
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--weather-start', help='天气回填起始日 YYYY-MM-DD')
    ap.add_argument('--weather-end', default=None)
    ap.add_argument('--dwgm', action='store_true', help='回填 DWGM (int041)')
    args = ap.parse_args()
 
    did = False
    if args.weather_start:
        n = backfill_weather(args.weather_start, args.weather_end)
        print(f"天气回填完成：{n} 行")
        did = True
    if args.dwgm:
        n = backfill_dwgm()
        print(f"DWGM 回填完成：{n} 行")
        did = True
    if not did:
        ap.print_help()
 
 
if __name__ == "__main__":
    main()
 

usage: ipykernel_launcher.py [-h] [--weather-start WEATHER_START]
                             [--weather-end WEATHER_END] [--dwgm]
ipykernel_launcher.py: error: unrecognized arguments: --f=c:\Users\phoeb\AppData\Roaming\jupyter\runtime\kernel-v3132f93e2bc57efcd07280e2863a89219aec440a0.json


SystemExit: 2

In [ ]:
import sqlite3
import pandas as pd
import os
import datetime

# 保持项目统一的路径保护
DB_PATH = os.path.join("data", "processed", "east_coast_gas.db")
STTM_CSV_URL = "https://nemweb.com.au/Reports/CURRENT/STTM/int651_v1_ex_ante_market_price_rpt_1.csv"

# 业务规则：只回填此日期【之前】的数据（不包含该日）
CUTOFF_DATE = '2026-06-15'

def backfill_sttm_prices():
    print("📥 正在直连读取 NEMWEB STTM 历史价格...")
    
    # 1. 内存级读取在线 CSV
    df = pd.read_csv(STTM_CSV_URL)
    
    # 2. 标准化列名（防首尾空格和大小写暗坑）
    df.columns = [str(col).strip().lower() for col in df.columns]
    
    # 3. 核心清洗：日期转换为标准 YYYY-MM-DD
    if 'gas_date' in df.columns:
        df['gas_date'] = pd.to_datetime(df['gas_date']).dt.strftime('%Y-%m-%d')
        
    # 4. 强制转换价格字段为浮点数
    if 'ex_ante_market_price' in df.columns:
        df['ex_ante_market_price'] = pd.to_numeric(df['ex_ante_market_price'], errors='coerce')


    original_count = len(df)
    df = df[df['gas_date'] < CUTOFF_DATE]
    filtered_count = len(df)
    
    print(f"✂️ 时间切片完成：剔除了 {original_count - filtered_count} 条近期数据，保留 {filtered_count} 条历史记录（严格早于 {CUTOFF_DATE}）。")
        
    # 5. 注入数据流审计元数据
    ingested_at = datetime.datetime.now(datetime.timezone.utc).isoformat()
    source_file = "int651_v1_ex_ante_market_price_rpt_1.csv"
    
    # 6. 建立数据库连接，执行高效批量覆盖写入 (INSERT OR REPLACE)
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    
    success_count = 0
    
    print("⚙️  正在将截断后的历史价格回填至 sttm_prices 表 (适配窄表结构)...")
    for _, row in df.iterrows():
        # 1. 提取 CSV 的真实数据 (CSV 的列名依然是 hub_identifier 和 ex_ante_market_price)
        gas_date = row.get('gas_date')
        hub_id = row.get('hub_identifier')  # 注意：从 CSV 取值依然要用 CSV 自己的列名
        price_val = row.get('ex_ante_market_price')
        
        if pd.isna(gas_date) or pd.isna(hub_id) or pd.isna(price_val):
            continue
            
        # 2. 完美适配你的真实表结构！
        # 增加固定的 price_type = 'ex_ante'
        # 把价格写进统一的 price 列
        cursor.execute('''
            INSERT OR REPLACE INTO sttm_prices 
            (gas_date, hub, price_type, price, source_file, ingested_at) 
            VALUES (?, ?, ?, ?, ?, ?)
        ''', (gas_date, hub_id, 'Ex-Ante', float(price_val), source_file, ingested_at))
        
        success_count += 1
        
    conn.commit()
    conn.close()
    
    print(f"✅ 回填大成功！已成功将 {success_count} 条 STTM 历史价格灌入数据库。它现在可以光荣退休了！")

if __name__ == "__main__":
    backfill_sttm_prices()

📥 正在直连读取 NEMWEB STTM 历史价格...
✂️ 时间切片完成：剔除了 6 条近期数据，保留 15 条历史记录（严格早于 2026-06-15）。
⚙️  正在将截断后的历史价格回填至 sttm_prices 表...


OperationalError: table sttm_prices has no column named ex_ante_market_price